# Dueling DQN: separate value and advantage

A dueling network combines two streams: $$Q(s,a)=V(s)+A(s,a)-\frac{1}{|\mathcal A|}\sum_{a'}A(s,a').$$ Here $V$ is state value, $A$ action advantage, and $|\mathcal A|$ the action count. Centering makes the decomposition identifiable.

In [ ]:
from collections import deque

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn

ENV_ID = "CartPole-v1"
TOTAL_TIMESTEPS = 10_000
LEARNING_RATE = 3e-4
GAMMA = 0.99
BUFFER_SIZE = 10_000
BATCH_SIZE = 64
LEARNING_STARTS = 500
TRAIN_FREQ = 4
TARGET_UPDATE_INTERVAL = 250
EPSILON_START = 1.0
EPSILON_END = 0.01
EXPLORATION_STEPS = 2_000
SEED = 7

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
env = gym.make(ENV_ID)
observation_dim = int(np.prod(env.observation_space.shape))
action_dim = env.action_space.n

## 1. Build online and target networks

The target begins as an exact, non-optimized copy of the online network.

In [ ]:
class DuelingNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(nn.Linear(observation_dim, 128), nn.ReLU())
        self.value = nn.Sequential(nn.Linear(128, 128), nn.ReLU(), nn.Linear(128, 1))
        self.advantage = nn.Sequential(nn.Linear(128, 128), nn.ReLU(), nn.Linear(128, action_dim))
    def forward(self, observations):
        features = self.features(observations)
        value, advantage = self.value(features), self.advantage(features)
        return value + advantage - advantage.mean(dim=1, keepdim=True)

q_network = DuelingNetwork().to(device)
target_network = DuelingNetwork().to(device)
target_network.load_state_dict(q_network.state_dict())
target_network.eval()
optimizer = torch.optim.Adam(q_network.parameters(), lr=LEARNING_RATE)

## 2. Explore and replay transitions

Epsilon-greedy behavior collects transitions. Only `terminated` disables bootstrapping; `truncated` still has a successor value.

In [ ]:
replay_buffer = deque(maxlen=BUFFER_SIZE)

def epsilon_at(step):
    fraction = min(step / EXPLORATION_STEPS, 1.0)
    return EPSILON_START + fraction * (EPSILON_END - EPSILON_START)

def select_action(observation, step, deterministic=False):
    if not deterministic and rng.random() < epsilon_at(step):
        return int(rng.integers(action_dim))
    observation = torch.as_tensor(observation, dtype=torch.float32, device=device).unsqueeze(0)
    with torch.no_grad():
        return int(q_network(observation).argmax(dim=1).item())

def sample_batch():
    indices = rng.choice(len(replay_buffer), BATCH_SIZE, replace=False)
    states, actions, rewards, next_states, terminals = zip(*(replay_buffer[index] for index in indices))
    return (
        torch.as_tensor(np.asarray(states), dtype=torch.float32, device=device),
        torch.as_tensor(actions, dtype=torch.int64, device=device),
        torch.as_tensor(rewards, dtype=torch.float32, device=device),
        torch.as_tensor(np.asarray(next_states), dtype=torch.float32, device=device),
        torch.as_tensor(terminals, dtype=torch.float32, device=device),
    )

## 3. Optimize the Bellman target

Dueling changes the architecture, while the target remains $y=R+\gamma(1-d)\max_{a'}Q_{\theta^-}(S',a')$.

In [ ]:
def train_step():
    states, actions, rewards, next_states, terminals = sample_batch()
    predictions = q_network(states).gather(1, actions.unsqueeze(1)).squeeze(1)
    with torch.no_grad():
        next_values = target_network(next_states).max(dim=1).values
        targets = rewards + GAMMA * (1 - terminals) * next_values
    loss = nn.functional.huber_loss(predictions, targets)
    optimizer.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(q_network.parameters(), 10.0); optimizer.step()
    return loss.item()

## 4. Connect collection and learning

Learning begins after replay has data, runs every few steps, and periodically copies online parameters to the target.

In [ ]:
def train(total_timesteps):
    episode_returns, losses = [], []
    episode_return = 0.0
    observation, _ = env.reset(seed=SEED)
    for step in range(1, total_timesteps + 1):
        action = select_action(observation, step - 1)
        next_observation, reward, terminated, truncated, _ = env.step(action)
        replay_buffer.append((observation, action, reward, next_observation, terminated))
        episode_return += reward
        if step >= LEARNING_STARTS and step % TRAIN_FREQ == 0 and len(replay_buffer) >= BATCH_SIZE:
            losses.append(train_step())
        if step % TARGET_UPDATE_INTERVAL == 0:
            target_network.load_state_dict(q_network.state_dict())
        if terminated or truncated:
            episode_returns.append(episode_return)
            episode_return = 0.0
            observation, _ = env.reset()
        else:
            observation = next_observation
    return episode_returns, losses

episode_returns, losses = train(TOTAL_TIMESTEPS)
env.close()

In [ ]:
returns = np.asarray(episode_returns)
window = min(10, len(returns))
moving_average = np.convolve(returns, np.ones(window) / window, mode="valid")
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(returns, alpha=0.3, label="Episode return")
axes[0].plot(np.arange(window - 1, len(returns)), moving_average, label=f"{window}-episode average")
axes[0].set(xlabel="Episode", ylabel="Return", title="Dueling DQN on CartPole-v1")
axes[0].legend()
axes[1].plot(losses)
axes[1].set(xlabel="Gradient update", ylabel="Huber loss", title="Training loss")
for axis in axes: axis.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## 5. Evaluate the greedy policy

This opens a separate rendered environment and disables exploration for 5 episodes.

In [ ]:
evaluation_env = gym.make(ENV_ID, render_mode="human")
evaluation_returns = []
q_network.eval()
try:
    for episode in range(5):
        observation, _ = evaluation_env.reset(seed=10 + episode)
        total_reward = 0.0
        done = False
        while not done:
            action = select_action(observation, TOTAL_TIMESTEPS, deterministic=True)
            observation, reward, terminated, truncated, _ = evaluation_env.step(action)
            total_reward += reward
            done = terminated or truncated
        evaluation_returns.append(total_reward)
finally:
    evaluation_env.close()
print("Episode returns:", evaluation_returns)